# 04. Weather and the information cut-off

**Where we are:** notebook 03 showed that weather drives most of what the calendar cannot:
temperature through heating, and sunshine through embedded solar, which removes about 1 MW of
`ND` for every MW it generates.

**The trap:** the weather that actually happened is not known a day ahead. A model trained and
tested on actual weather would look far better than any real forecast could be. NESO had only
weather forecasts when it published, so our model must be tested on weather forecasts too.

**What this notebook does:**

1. Fixes exactly what our forecast is allowed to know, and when.
2. Checks a live source of demand, so the lag features are honest.
3. Downloads actual weather (for training) and archived day-ahead forecasts (for testing).
4. Measures how far the forecasts are from what happened, and what that costs in MW.

**Background reading**
- [Open-Meteo Previous Runs API](https://open-meteo.com/en/docs/previous-runs-api): archived forecasts at fixed lead times
- [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api): ERA5 reanalysis
- [Forecasting: Principles and Practice, section 7.6, forecasting with regression](https://otexts.com/fpp3/forecasting-regression.html): why predictors must themselves be known in advance


## Step 1. Set up


In [1]:
import time
from pathlib import Path

import holidays
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
TEST_START = pd.Timestamp("2024-06-01", tz="Europe/London")

BLUE, ORANGE = "#2a78d6", "#eb6834"
plt.rcParams.update({
    "figure.dpi": 90,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e4e4e0",
    "lines.linewidth": 2,
})

demand = pd.read_parquet(PROCESSED / "demand.parquet")


def get_json(url, params, tries=5):
    """GET with a pause and retry when the API says too many requests (HTTP 429)."""
    for _ in range(tries):
        response = requests.get(url, params=params, timeout=300)
        if response.status_code != 429:
            response.raise_for_status()
            return response.json()
        time.sleep(65)
    response.raise_for_status()


## Step 2. The forecast origin

**What:** state, as code, the moment our forecast is made.

**Why:** every feature has to pass one test: was it known at that moment? Writing the origin
down once, as a function, means every later notebook uses the same rule.

Notebook 02 found NESO publishes between 08:33 and 10:33 UTC on the day before. Our origin is
**08:30 UTC on the day before the target day**, which is never later than NESO.

The target day runs from local midnight to local midnight, so from the origin the forecast
covers lead times of roughly 15 to 40 hours:


In [2]:
ORIGIN_TIME = pd.Timedelta(hours=8, minutes=30)


def forecast_origin(target_date):
    """UTC moment the forecast for a local target date is made: 08:30 UTC the day before."""
    day_before = pd.Timestamp(target_date) - pd.Timedelta(days=1)
    return (day_before + ORIGIN_TIME).tz_localize("UTC")


for target in ["2025-01-15", "2025-07-15"]:
    origin = forecast_origin(target)
    first = pd.Timestamp(target).tz_localize("Europe/London").tz_convert("UTC")
    last_end = (pd.Timestamp(target) + pd.Timedelta(days=1)).tz_localize("Europe/London").tz_convert("UTC")
    print(f"target {target}: origin {origin:%Y-%m-%d %H:%M} UTC, "
          f"lead {(first - origin) / pd.Timedelta(hours=1):.1f} to {(last_end - origin) / pd.Timedelta(hours=1):.1f} hours")


target 2025-01-15: origin 2025-01-14 08:30 UTC, lead 15.5 to 39.5 hours
target 2025-07-15: origin 2025-07-14 08:30 UTC, lead 14.5 to 38.5 hours


## Step 3. Is recent demand known at the origin?

**What:** download Elexon's *initial* national demand outturn (`INDO`) and check when it is
published and how close it is to `ND`.

**Why:** NESO's historic demand files are published weeks after the fact, so they cannot be a
live input. Notebook 03 planned to use only demand from two or more days back for that reason.
But GB demand is also published within minutes by Elexon, the market's settlement body. If that
live figure matches `ND`, the forecast can also use demand from the morning of the day before.

**Why the same source for training:** `INDO` is an early estimate that NESO later revises into
`ND`. If the model learns from revised `ND` lags but is fed early `INDO` lags in use, the inputs
would differ between training and testing. Building lag features from `INDO` in both removes
that mismatch.

**Further reading:** [Elexon Insights API](https://developer.data.elexon.co.uk), dataset
`INDO` (initial national demand outturn)


In [3]:
INDO_FILE = RAW / "elexon_indo.parquet"
ELEXON = "https://data.elexon.co.uk/bmrs/api/v1/demand/outturn"

if not INDO_FILE.exists():
    chunks = []
    start, end = pd.Timestamp("2018-01-01"), pd.Timestamp.now().normalize() - pd.Timedelta(days=1)
    while start <= end:
        stop = min(start + pd.Timedelta(days=27), end)  # the API allows 28 days per request
        response = requests.get(ELEXON, params={
            "settlementDateFrom": f"{start:%Y-%m-%d}", "settlementDateTo": f"{stop:%Y-%m-%d}", "format": "json",
        }, timeout=60)
        response.raise_for_status()
        chunks.append(pd.DataFrame(response.json()["data"]))
        start = stop + pd.Timedelta(days=1)
        time.sleep(0.2)
    pd.concat(chunks, ignore_index=True).to_parquet(INDO_FILE, index=False)

indo = pd.read_parquet(INDO_FILE)
indo["start_utc"] = pd.to_datetime(indo["startTime"], utc=True)
indo["publish_utc"] = pd.to_datetime(indo["publishTime"], utc=True)
indo = indo.rename(columns={"settlementDate": "Date", "settlementPeriod": "Settlement_Period",
                            "initialDemandOutturn": "INDO"})
print(indo.shape, indo["Date"].min(), "to", indo["Date"].max())
print("repeated half-hours:", int(indo.duplicated("start_utc").sum()))

delay = (indo["publish_utc"] - (indo["start_utc"] + pd.Timedelta(minutes=30))).dt.total_seconds() / 60
print("minutes from end of period to publication:")
print(delay.describe().round(1).to_string())
print(f"published more than 30 minutes after the period ends: {(delay > 30).mean():.4%}")


(154722, 8) 2018-01-01 to 2026-11-11
repeated half-hours: 0
minutes from end of period to publication:
count    154722.0
mean          0.2
std           5.9
min           0.0
25%           0.0
50%           0.0
75%           0.0
max        1193.0
published more than 30 minutes after the period ends: 0.1713%


In [4]:
indo = indo.drop_duplicates("start_utc", keep="last")
both = demand[["start_utc", "ND"]].merge(indo[["start_utc", "INDO"]], on="start_utc", how="inner")
both["gap"] = both["INDO"] - both["ND"]
both["year"] = both["start_utc"].dt.year

summary = both.groupby("year").agg(
    rows=("gap", "size"),
    mean_gap=("gap", "mean"),
    mean_abs_gap=("gap", lambda g: g.abs().mean()),
    corr=("INDO", lambda s: s.corr(both.loc[s.index, "ND"])),
)
summary.round(3)


,rows,mean_gap,mean_abs_gap,corr
year,,,,
2018,17376,-62.695,75.627,1.0
2019,17424,-51.311,70.118,1.0
2020,17398,-41.323,62.145,1.0
2021,17280,-36.811,53.477,1.0
2022,17518,-35.049,55.863,1.0
2023,17518,-36.413,59.034,1.0
2024,17568,-35.585,58.040,1.0
2025,17520,-36.963,59.364,1.0
2026,14014,-25.925,47.876,1.0


**Found:**

- `INDO` is published as each half-hour ends: the median delay is zero minutes. Only 0.17% of
  periods came out more than 30 minutes late (the longest delay was about 20 hours). So at the origin, demand is
  known up to the morning of the day before, not only up to two days before.
- It tracks `ND` closely: the correlation rounds to 1.00, and the mean absolute gap is 48 to 76 MW,
  about 0.2% of demand. `INDO` runs about 36 MW below `ND` in recent years (63 MW in 2018), which
  is the size of the revisions NESO makes later.

**Decision:**
- Lag features come from `INDO`, in training and in testing, so the inputs mean the same thing
  in both. The target stays `ND`.
- The forecast may use `INDO` up to 08:00 UTC on the day before (Step 11 explains the margin).
  This replaces the plan's "two days or more" rule, which was stricter than needed. Notebook 03
  found yesterday's demand (correlation 0.915) only a little more informative than last week's
  (0.911), so Day 6 will check how much the extra morning of data helps.


## Step 4. Weather locations

**What:** choose the places to take weather from.

**Why several places:** GB demand is spread across the country, and so is embedded solar and
wind. One city would miss weather fronts that cover only part of the island.

**Choice:** the eight largest cities in Great Britain across England, Scotland and Wales, plus
Plymouth and Norwich, because the south-west and East Anglia have a lot of solar farms but no
big city. Coordinates and populations come from Open-Meteo's geocoding service (GeoNames).

**Why equal weights:** a population-weighted average sounds better, but GeoNames gives
populations of the city itself, not its urban area (Manchester appears as 0.57 million, London
as 9 million), which would give London two-thirds of the weight. An equal-weight mean of ten
spread-out places is simpler and easier to defend. Day 6 can test whether per-location values
help.


In [5]:
CITIES = ["London", "Birmingham", "Manchester", "Leeds", "Glasgow", "Bristol",
          "Newcastle upon Tyne", "Cardiff", "Plymouth", "Norwich"]
LOCATIONS_FILE = RAW / "weather_locations.csv"

if not LOCATIONS_FILE.exists():
    rows = []
    for city in CITIES:
        results = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                               params={"name": city, "count": 5, "countryCode": "GB"}, timeout=30).json()["results"]
        best = results[0]
        rows.append({"city": city, "latitude": best["latitude"], "longitude": best["longitude"],
                     "population": best.get("population"), "nation": best.get("admin1")})
    pd.DataFrame(rows).to_csv(LOCATIONS_FILE, index=False)

locations = pd.read_csv(LOCATIONS_FILE)
locations["population share"] = (locations["population"] / locations["population"].sum()).round(3)
locations


,city,latitude,longitude,population,nation,population share
0,London,51.50853,-0.12574,8961989,England,0.669
1,Birmingham,52.48142,-1.89983,1157603,England,0.086
2,Manchester,53.48095,-2.23743,568996,England,0.042
3,Leeds,53.79648,-1.54785,536280,England,0.040
4,Glasgow,55.86515,-4.25763,626410,Scotland,0.047
5,Bristol,51.45523,-2.59665,479024,England,0.036
6,Newcastle upon Tyne,54.97328,-1.61396,300125,England,0.022
7,Cardiff,51.48000,-3.18000,372089,Wales,0.028
8,Plymouth,50.37153,-4.14305,260203,England,0.019
9,Norwich,52.62783,1.29834,143135,England,0.011


## Step 5. Actual weather: ERA5 reanalysis

**What:** hourly temperature, solar radiation and wind speed at each location from 2018, from
ERA5 via Open-Meteo's archive.

**What reanalysis is:** a weather model rerun over the past with all the observations that were
made, giving a consistent best estimate of what the weather actually was, on a 0.25° grid
(roughly 25 km). It is published a few days late.

**Why use it for training:** archived day-ahead forecasts only go back to early 2024 (Step 6),
which is too short to train on. So the model learns from actual weather and is tested on
forecast weather. Step 9 measures what that costs.

**Rate limits and terms:** Open-Meteo's free API allows 600 weighted calls a minute and 10,000 a
day, for non-commercial use, and a year of ten locations counts as a few hundred calls. So the
download goes one year at a time with a pause, and retries after a minute if refused. Files are
saved, so this only happens once. The data is CC BY 4.0: credit Open-Meteo and ERA5 (Copernicus).

**Note on radiation:** Open-Meteo gives `shortwave_radiation` as the mean over the hour *before*
the timestamp; temperature and wind are values *at* the timestamp. Step 7 lines this up.


In [6]:
VARIABLES = ["temperature_2m", "shortwave_radiation", "wind_speed_10m"]
LAT = ",".join(locations["latitude"].astype(str))
LON = ",".join(locations["longitude"].astype(str))


def to_long(payload, suffix=""):
    """Open-Meteo multi-location response to one long table."""
    frames = []
    for city, item in zip(locations["city"], payload):
        hourly = pd.DataFrame(item["hourly"])
        hourly.columns = [c.replace(suffix, "") for c in hourly.columns]
        hourly["city"] = city
        frames.append(hourly)
    out = pd.concat(frames, ignore_index=True)
    out["time"] = pd.to_datetime(out["time"]).dt.tz_localize("UTC")
    return out


ERA5_FILE = RAW / "weather_era5.parquet"
if not ERA5_FILE.exists():
    parts = []
    for year in range(2018, pd.Timestamp.now().year + 1):
        payload = get_json("https://archive-api.open-meteo.com/v1/archive", {
            "latitude": LAT, "longitude": LON, "hourly": ",".join(VARIABLES), "models": "era5",
            "start_date": f"{year}-01-01", "end_date": min(f"{year}-12-31", f"{pd.Timestamp.now():%Y-%m-%d}"),
            "timezone": "GMT",
        })
        parts.append(to_long(payload))
        time.sleep(30)
    pd.concat(parts, ignore_index=True).dropna(subset=VARIABLES, how="all").to_parquet(ERA5_FILE, index=False)

era5 = pd.read_parquet(ERA5_FILE).dropna(subset=VARIABLES)
print(era5.shape, era5["time"].min(), "to", era5["time"].max())


(775680, 5) 2018-01-01 00:00:00+00:00 to 2026-11-06 23:00:00+00:00


## Step 6. Day-ahead weather forecasts

**What:** the same variables as forecast in advance, from Open-Meteo's Previous Runs API.

**Which lead time, and why it matters.** The API offers `_previous_day1` ("the value predicted
24 hours before valid time") and `_previous_day2` (48 hours before). The project plan assumed
`_previous_day1`. That would leak: for a target half-hour at 18:00 on day D, the run from 24
hours earlier started at 18:00 on D−1, nearly ten hours *after* our 08:30 origin.

`_previous_day2` comes from a run started 48 hours before valid time. For the latest target
hour (23:30 on day D) that is 23:30 on D−2, nine hours before the origin, which leaves time for
the model run to finish and be published. So `_previous_day2` is always fair, but is a little
older than the forecast NESO could have used. That puts our model at a small disadvantage,
which is the safe direction.

The API also says most models are archived from January 2024, so this data starts there.


In [7]:
FORECAST_FILE = RAW / "weather_forecast_day2.parquet"
SUFFIX = "_previous_day2"

if not FORECAST_FILE.exists():
    parts = []
    end = pd.Timestamp.now().normalize() - pd.Timedelta(days=1)
    for start in pd.date_range("2024-01-01", end, freq="QS"):
        stop = min(start + pd.offsets.QuarterEnd(0), end)
        payload = get_json("https://previous-runs-api.open-meteo.com/v1/forecast", {
            "latitude": LAT, "longitude": LON, "hourly": ",".join(v + SUFFIX for v in VARIABLES),
            "start_date": f"{start:%Y-%m-%d}", "end_date": f"{stop:%Y-%m-%d}", "timezone": "GMT",
        })
        parts.append(to_long(payload, suffix=SUFFIX))
        time.sleep(20)
    pd.concat(parts, ignore_index=True).to_parquet(FORECAST_FILE, index=False)

forecast = pd.read_parquet(FORECAST_FILE)
for variable in VARIABLES:
    first = forecast.loc[forecast[variable].notna(), "time"].min()
    print(f"{variable:22s} first available {first:%Y-%m-%d %H:%M}")
forecast = forecast.dropna(subset=VARIABLES)
print(forecast.shape, forecast["time"].min(), "to", forecast["time"].max())


temperature_2m         first available 2024-02-05 00:00
shortwave_radiation    first available 2024-03-07 17:00
wind_speed_10m         first available 2024-02-05 00:00
(235030, 5) 2024-03-07 17:00:00+00:00 to 2026-11-11 23:00:00+00:00


**Found:** archived day-ahead forecasts start on 5 February 2024 for temperature and wind and on
7 March 2024 for radiation. That confirms the test window start of 1 June 2024 leaves about three
months of overlap before it (March to May 2024), which Step 10 uses.


## Step 7. From hourly to half-hourly

**What:** average the ten locations, then turn hourly values into one value per settlement period.

**How:**
- **Temperature and wind** are values at the hour, so each half-hour gets the value at its
  midpoint (15 minutes past its start), by straight-line interpolation between hours.
- **Radiation** is the mean over the hour before the timestamp. A half-hour starting at 12:00
  or 12:30 lies inside the hour that ends at 13:00, so it takes the 13:00 value.

**Why care about a half-hour:** solar output rises and falls quickly around sunrise and sunset.
Shifting radiation by an hour would put sunlight in the wrong periods and blur exactly the
effect we need.


In [8]:
def to_half_hours(hourly):
    """Equal-weight mean over locations, then one row per settlement period (UTC start)."""
    mean = hourly.groupby("time")[VARIABLES].mean()
    grid = pd.date_range(mean.index.min(), mean.index.max() - pd.Timedelta(minutes=30), freq="30min")
    out = pd.DataFrame(index=grid)

    midpoints = grid + pd.Timedelta(minutes=15)
    for column in ["temperature_2m", "wind_speed_10m"]:
        series = mean[column].reindex(mean.index.union(midpoints)).interpolate("time")
        out[column] = series.reindex(midpoints).values

    covering_hour = grid.floor("h") + pd.Timedelta(hours=1)
    out["shortwave_radiation"] = mean["shortwave_radiation"].reindex(covering_hour).values
    return out.rename_axis("start_utc").reset_index()


weather_actual = to_half_hours(era5)
weather_forecast = to_half_hours(forecast)
print(weather_actual.shape, weather_forecast.shape)
weather_forecast.iloc[20:26]


(155134, 4) (47004, 4)


,start_utc,temperature_2m,wind_speed_10m,shortwave_radiation
20,2024-03-08 03:00:00+00:00,4.5150,20.065,0.0
21,2024-03-08 03:30:00+00:00,4.4050,19.995,0.0
22,2024-03-08 04:00:00+00:00,4.2975,19.935,0.0
23,2024-03-08 04:30:00+00:00,4.1925,19.885,0.0
24,2024-03-08 05:00:00+00:00,4.1475,19.950,0.0
25,2024-03-08 05:30:00+00:00,4.1625,20.130,0.0


## Step 8. Save the weather tables

**What:** save both half-hourly tables, and Elexon's live demand, as Parquet for the modelling notebooks.


In [9]:
weather_actual.to_parquet(PROCESSED / "weather_actual.parquet", index=False)
weather_forecast.to_parquet(PROCESSED / "weather_forecast.parquet", index=False)
indo[["start_utc", "Date", "Settlement_Period", "INDO", "publish_utc"]].to_parquet(PROCESSED / "indo.parquet", index=False)
for name in ["weather_actual", "weather_forecast", "indo"]:
    print(name, pd.read_parquet(PROCESSED / f"{name}.parquet").shape)


weather_actual (155134, 4)
weather_forecast (47004, 4)
indo (154722, 5)
